# RAG Pipeline: PDF Chunking + Retrieval

**Tools used:** Visual Studio Code, Python, FAISS, LangChain, OpenAI API

**What this notebook does, in plain English:**
1. Load a few PDF files
2. Cut them into small overlapping "chunks" of text
3. Turn each chunk into a list of numbers (an "embedding") that captures its meaning, using OpenAI's embedding model
4. Store those embeddings in a FAISS vector database (a fast similarity search index)
5. Given a question, find the chunks whose meaning is closest to the question
6. Hand those chunks to an OpenAI chat model and ask it to answer using only that retrieved context

**Before running:** drop 2-3 PDF files into the `data/` folder next to this notebook, and put your OpenAI API key in a `.env` file (copy `.env.example` to `.env` and fill it in).

Run the cells top to bottom with Shift+Enter. Read the markdown before each cell -- it explains what's about to happen and why.

## Step 0 - Imports and API key

This loads your `OPENAI_API_KEY` from the `.env` file into the environment so the OpenAI client can find it automatically. If this cell errors with a missing key, double check `.env` exists in this folder and has the line `OPENAI_API_KEY=sk-...`.

In [ ]:
import glob
from dotenv import load_dotenv

load_dotenv()  # reads .env and sets OPENAI_API_KEY as an environment variable
print("Setup complete.")

## Step 1 - Load the PDFs

`PyPDFLoader` reads one PDF and returns a list of `Document` objects -- one per page. Each `Document` has `.page_content` (the text) and `.metadata` (like which file and page it came from, useful later for citing sources).

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_paths = glob.glob("data/*.pdf")
print(f"Found {len(pdf_paths)} PDF(s): {pdf_paths}")

docs = []
for path in pdf_paths:
    docs.extend(PyPDFLoader(path).load())

print(f"Loaded {len(docs)} pages total.")
print("\n--- Sample of first page's text ---")
print(docs[0].page_content[:500] if docs else "No PDFs found -- add some to the data/ folder.")

## Step 2 - Chunk the documents

Whole pages are usually too big and too unfocused to embed well. `RecursiveCharacterTextSplitter` breaks the text into ~1000-character chunks, with 150 characters of overlap between consecutive chunks so we don't cut a sentence or idea in half at a chunk boundary.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs)

print(f"Split {len(docs)} pages into {len(chunks)} chunks.")
print("\n--- Sample chunk ---")
print(chunks[0].page_content if chunks else "No chunks -- check Step 1.")
print("\nMetadata:", chunks[0].metadata if chunks else None)

## Step 3 - Generate embeddings and build the FAISS index

We use OpenAI's `text-embedding-3-small` model to turn each chunk of text into a vector of 1536 numbers (this makes API calls and costs a small amount -- a few cents for a handful of PDFs). FAISS stores these vectors and lets us quickly find the ones closest to a new query vector.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

print(f"Built FAISS index with {vectorstore.index.ntotal} vectors, saved to faiss_index/")

## Step 4 - Run a retrieval query (semantic search only)

Before involving the chat model at all, let's see retrieval by itself: given a question, which chunks does FAISS think are most relevant? This is the core of "semantic search" -- it matches by *meaning*, not by exact keyword overlap.

Edit `sample_query` below to something relevant to your own PDFs.

In [ ]:
sample_query = "What is this document about?"  # <-- change this to a question about your PDFs

results = vectorstore.similarity_search(sample_query, k=3)

for i, r in enumerate(results, 1):
    print(f"--- Match {i} (source: {r.metadata.get('source')}, page {r.metadata.get('page')}) ---")
    print(r.page_content[:400])
    print()

## Step 5 - Full RAG: retrieve + OpenAI generates an answer

Now we chain it together: retrieve the top matching chunks, then hand them to an OpenAI chat model as context and ask it to answer the question *using only that context*. This is what makes it "retrieval-augmented generation" -- the model isn't answering from memory, it's answering from the specific documents you gave it.

In [2]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
)

def ask(question: str):
    result = qa_chain.invoke({"query": question})
    print("QUESTION:", question)
    print("\nANSWER:", result["result"])
    print("\nSOURCES:")
    for doc in result["source_documents"]:
        print(f"  - {doc.metadata.get('source')} (page {doc.metadata.get('page')})")
    return result

_ = ask(sample_query)

NameError: name 'vectorstore' is not defined

## Step 6 - Try your own questions

This is the part you'll want sample outputs from for your submission. Ask 3-5 questions:
- A couple that are clearly answerable from the PDFs
- One that is *not* in the PDFs at all, to see how the model handles missing information

Copy/paste the `ask(...)` line and change the question for each one, so the outputs stay in the notebook for your submission.

In [ ]:
_ = ask("Replace this with your first real question about your documents")

In [ ]:
_ = ask("Replace this with a second question")

In [ ]:
_ = ask("Ask something that is NOT covered in your PDFs, to test how it handles missing info")

## Step 7 - Notes for your analysis report

For the write-up the assignment asks for, it helps to note:
- Which PDFs you used and roughly how long they are
- Chunk size / overlap chosen (1000 / 150 here) and whether that felt right
- How many chunks were created
- 3-5 example questions and answers (captured above)
- How it behaved on the out-of-scope question (did it say it didn't know, or did it guess?)
- Anything you'd change if you did it again (different chunk size, different `k`, different embedding model)